# Juntar archivos en uno solo
***

El objetivo de este notebook es juntar todos los netcdf de todas las secciones ya filtradas, y unirlas en un único archivo, de forma que se concatenan los perfiles (N_PROF), y el número de niveles es el máximo entre todos.

Este notebok lee de ./Data/corrected_sections_filtrado/ y devuelve una matriz de datos globales y multitemporales en la carpeta ./Data/join

Como adición, también calcula la capacidad calorifica a presión constante y la densidad a partir de los datos de salinidad. Así para cada perfil y cada presion hay un dato de temperatura, salinidad, capacidad calorífica a presión constante y densidad.

In [ ]:
# Package for file manipulation
import os
import glob

# Packages for data manipulation
import numpy as np
import matplotlib.pyplot as plt
import xarray as xr

# Package for progress bar
import tqdm as tqdm

# Package for density and Cp calculation
import seawater as sw

# Import a function from ReadOriginalData.py
from ReadOriginalData import vars_coords_interest

C:\Users\ismae\AppData\Local\Temp\ipykernel_9112\1922633165.py:8: UserWarning: The seawater library is deprecated! Please use gsw instead.
  import seawater as sw


Vamos a leer todos los archivos, y vamos a quedarnos con el número total de N_LEVELS y N_PROF. De forma que para cada variable tendremos un array N_PROF x N_LEVELS (En el caso del archivo con la presión interpolada, será N_PROF x max(presion_interp)).

Nos quedaremos con datos importantes como el section_id, latitud y longitud, fecha, y nombre del archivo para cada perfil, entre otros.

In [ ]:
path = "./Data/corrected_sections_filtrado/"
sections = sorted(glob.glob(path+'/*'))

lats = []
lons = []
dates = []
file_name = [] # List with the file name for each profile
file_names = [] # List with the name of the file for the name file
section_id = []
N_PROF = []

# To see the progress bar
verbose = True

In [ ]:
# See the sections
sections

['./Data/corrected_sections_filtrado\\A01',
 './Data/corrected_sections_filtrado\\A02',
 './Data/corrected_sections_filtrado\\A05',
 './Data/corrected_sections_filtrado\\A10',
 './Data/corrected_sections_filtrado\\A12',
 './Data/corrected_sections_filtrado\\A13.5',
 './Data/corrected_sections_filtrado\\A16',
 './Data/corrected_sections_filtrado\\A20',
 './Data/corrected_sections_filtrado\\A22',
 './Data/corrected_sections_filtrado\\I03',
 './Data/corrected_sections_filtrado\\I04',
 './Data/corrected_sections_filtrado\\I05',
 './Data/corrected_sections_filtrado\\I06',
 './Data/corrected_sections_filtrado\\I08',
 './Data/corrected_sections_filtrado\\I09S',
 './Data/corrected_sections_filtrado\\P01',
 './Data/corrected_sections_filtrado\\P02',
 './Data/corrected_sections_filtrado\\P03',
 './Data/corrected_sections_filtrado\\P06',
 './Data/corrected_sections_filtrado\\P10',
 './Data/corrected_sections_filtrado\\P14',
 './Data/corrected_sections_filtrado\\P15',
 './Data/corrected_sections_f

In [ ]:
# Iterate for each section
for section in sections:
    sec_prof = 0
    section_path = section + "/"
    files = sorted(os.listdir(section_path))

    # Iterate for each file    
    for file in files:
        print(file)
        try:
            assert file.endswith(".nc")
            ds = xr.open_dataset(section_path + file)
            vars_, coords, _ = vars_coords_interest(ds)

            N_PROF.append(np.max(ds.N_PROF.values))
            file_names.append(file)
            sec_prof += np.max(ds.N_PROF.values)

            # Iterate each profile
            for prof in range(np.max(ds.N_PROF.values)):
                section_id.append(section)
                lats.append(ds.latitude.values[prof])
                lons.append(ds.longitude.values[prof])
                file_name.append(file)
                dates.append(np.datetime64(ds.time.values[prof]) if "time" in ds.coords else np.datetime64(file[:4]))

            press_max = np.max(ds.pressure_interp)
            ds.close()

        except AssertionError:
            continue

1991_06MT18_1_ctd.nc
1994_06MT30_3_ctd.nc
1995_18HU95011_1_ctd.nc
2000_64PE20000926_ctd.nc
2014_35PK20140515_ctd.nc
2015_5_58GS20150410_ctd.nc
1994_06MT030_2_ctd.nc
1997_06MT039_3_ctd.nc
2011_06MT20110624_ctd.nc
2013_9_06M220130509_ctd.nc
2017_45CE20170427_ctd.nc
1981_1981_US008812.nc
1992_29HE06_1_ctd.nc
1998_31RBOACES24N_2_ctd.nc
2004_74DI20040404_ctd.nc
2010_74DI20100106_ctd.nc
2011_29AH20110128_ctd.nc
2015_2016_74EQ20151206_ctd.nc
2020_740H20200119_ctd.nc
2022_6_74EQ20220209_ctd.nc
1992_1993_06MT22_5_ctd.nc
2003_49NZ200311_4_ctd.nc
2009_740H20090307_ctd.nc
2011_33RO20110926_ctd.nc
2018_740H20180228_ctd.nc
1986_None_ctd.nc
1992_06AQANTX_4_ctd.nc
1994_06AQ19941123_ctd.nc
1999_06ANTXVI_2_ctd.nc
2000_2001_06ANTXVIII_3_ctd.nc
2002_2003_06AQ200211_2_ctd.nc
2005_06AQ20050122_ctd.nc
2006_06AQ20060825_ctd.nc
2007_2008_06AQ20071128_ctd.nc
2008_06AQ20080210_ctd.nc
2008_35MF20080207_ctd.nc
2014_2015_06AQ20141202_ctd.nc
1983_316N19831007_ctd.nc
1984_316N19840111_ctd.nc
1995_35A3CITHER3_2_ctd.nc

Definimos el N_PROF y N_LEVELS del dataset total.

In [5]:
N_PROF_total = np.sum(N_PROF)
N_LEVELS_max = int(press_max)

Vamos a hacer una concatenación en columnas de las variables en cada perfil. En este caso, como todos los archivos filtrados tienen el mismo rango en la presión, no hay que hacer padding.

El plan general es crear un diccionario que contiene la variable correspondiente para cada sección, para más tarde hacer una concatenación en la dimensión de los perfiles. Esto es más eficiente que crear directamente el array con todos los perfiles de todos los archivos, por problemas de memoria.

In [ ]:
temp_dict = {}
sal_dict = {}
ox_dict = {}
dens_dict = {}
cp_dict = {}

# Iterate for each section
for i, section in enumerate(sections):
    ctd_temperature = np.array([])
    ctd_salinity = np.array([])
    ctd_oxygen = np.array([])
    ctd_density = np.array([])
    ctd_cp = np.array([])

    section_path = section + "/"
    files = sorted(os.listdir(section_path))
    for idx, file in tqdm.tqdm(enumerate(files), total = len(files), desc = f"Section {section}") if verbose == True else enumerate(files):
            try:
                assert file.endswith(".nc")
                ds = xr.open_dataset(section_path + file)

                # Extract the value matrix
                temp = ds["ctd_temperature_filt"].to_numpy()
                sal = ds["ctd_salinity_filt"].to_numpy()
                ox = ds["ctd_oxygen_filt"].to_numpy() if "ctd_oxygen_filt" in ds.data_vars else np.full((np.max(ds.N_PROF.values), N_LEVELS_max + 1), np.nan)
                p = ds['pressure_interp'].to_numpy()

                # Calculate density and Cp
                dens = sw.dens(sal, temp, p)
                cp = sw.cp(sal, temp, p)

                # Iterate each profile
                for ip, profile in enumerate(range(np.max(ds.N_PROF.values))):
                    # Extract the data from the wanted profiles
                    temp_ = temp[profile].copy()
                    sal_ = sal[profile].copy()
                    ox_ = ox[profile].copy()
                    dens_ = dens[profile].copy()
                    cp_ = cp[profile].copy()

                    if idx == 0 and ip == 0:
                        ctd_temperature = temp_
                        ctd_salinity = sal_
                        ctd_oxygen = ox_
                        ctd_density = dens_
                        ctd_cp = cp_
                    else:
                        ctd_temperature = np.column_stack((ctd_temperature, temp_))
                        ctd_salinity = np.column_stack((ctd_salinity, sal_))
                        ctd_oxygen = np.column_stack((ctd_oxygen, ox_))
                        ctd_density = np.column_stack((ctd_density, dens_))
                        ctd_cp = np.column_stack((ctd_cp, cp_))
                ds.close()
            except AssertionError:
                continue

    temp_dict[section] = ctd_temperature
    sal_dict[section] = ctd_salinity
    ox_dict[section] = ctd_oxygen
    dens_dict[section] = ctd_density
    cp_dict[section] = ctd_cp



Section ./Data/corrected_sections_filtrado\SR03: 100%|██████████| 13/13 [01:12<00:00,  5.56s/it]


Para no tener que ejecutar este código todo el rato, a parte de concatenar los arrays, los guardamos.

In [9]:
ctd_temperature_filt = np.concatenate([temp_dict[key] for key in temp_dict.keys()], axis = 1)
np.save("./Data/arrays/ctd_temperature_filt.npy", ctd_temperature_filt)

ctd_salinity_filt = np.concatenate([sal_dict[key] for key in sal_dict.keys()], axis = 1)
np.save("./Data/arrays/ctd_salinity_filt.npy", ctd_salinity_filt)

ctd_oxygen_filt = np.concatenate([ox_dict[key] for key in ox_dict.keys()], axis = 1)
np.save("./Data/arrays/ctd_oxygen_filt.npy", ctd_oxygen_filt)

ctd_density_filt = np.concatenate([dens_dict[key] for key in dens_dict.keys()], axis = 1)
np.save("./Data/arrays/ctd_density_filt.npy", ctd_density_filt)

ctd_cp_filt = np.concatenate([cp_dict[key] for key in cp_dict.keys()], axis = 1)
np.save("./Data/arrays/ctd_cp_filt.npy", ctd_cp_filt)


Aquí se cargan.

In [ ]:
# Solo necesrio si interrumpi el proceso y grabé en formato npy
#ctd_temperature_filt = np.load("./Data/arrays/ctd_temperature_filt.npy")
#ctd_salinity_filt = np.load("./Data/arrays/ctd_salinity_filt.npy")
#ctd_oxygen_filt = np.load("./Data/arrays/ctd_oxygen_filt.npy")
# ctd_density_filt = np.load("./Data/arrays/ctd_density_filt.npy")
# ctd_cp_filt = np.load("./Data/arrays/ctd_cp_filt.npy")


Creamos el dataset con xarray para su posterior guardado como archivo **.nc**

In [10]:
filt = xr.Dataset(
    data_vars = dict(
        ctd_temperature_filt = (["N_PROF", "P"], ctd_temperature_filt.T, {
            "whp_name" : "CTDTMP",
            "whp_units" : "ITS-90",
            "standard name" : "sea_water_temperature",
            "units" : "degC",
            "reference_scale" : "ITS-90"
        }),
        ctd_salinity_filt = (["N_PROF", "P"], ctd_salinity_filt.T, {
            "whp_name" : "CTDSAL",
            "whp_units" : "PSS-78",
            "standard name" : "sea_water_practical_salinity",
            "units" : "1",
            "reference_scale" : "PSS-78"
        }),
        ctd_oxygen_filt = (["N_PROF", "P"], ctd_oxygen_filt.T, {
            "whp_name" : "CTDOXY",
            "whp_units" : "UMOL/KG",
            "standard name" : "moles_of_oxygen_per_unit_mass_in_sea_water",
            "units" : "umol/kg"
        }),
        ctd_density_filt = (["N_PROF", "P"], ctd_density_filt.T, {
    "standard_name" : "sea_water_density",
    "units" : "kg/m^3"
}),
        ctd_cp_filt = (["N_PROF", "P"], ctd_cp_filt.T, {
    "standard_name" : "sea_water_specific_heat_capacity",
    "units" : "J/(kg C)"
})),
    coords = dict(
        time = (["N_PROF"], dates),
        section_id = (["N_PROF"], section_id),
        file_name = (["N_PROF"], file_name),
        latitude = (["N_PROF"], lats),
        longitude = (["N_PROF"], lons)        ,
        pressure = (["P"], np.arange(0, N_LEVELS_max + 1, 1))
    )
)

filt.to_netcdf("./Data/join/total_filt.nc")